# 01 — Train Full Pretrained Model (Θ₀) + Generate Splits

**Pipeline role:** Canonical first notebook. Produces:
1. One pretrained checkpoint per seed (`pretrain_dir/<experiment_name>_seed<N>.pt`)
2. Per-class forget/retain split JSON files for every `(seed, forget_class)` pair
3. A `split_summary.csv` for auditing

**No unlearning or Oracle retraining is performed here.**

## How to use
1. **Cell 0** — clones the repo from GitHub and installs dependencies (run once).
2. **Cell 1** — set `DATASET = "cifar10"` or `DATASET = "cifar100"`.  That is the only edit needed.
3. Run all remaining cells.

---
## Cell 0 — Setup: clone repo & install dependencies

Clones `https://github.com/tiensinh2/CMF_UNLearning_Posthoc.git` into the
current working directory (skipped if already present), installs Python
dependencies, and adds the repo root to `sys.path`.

> Run this cell **once** per environment (Colab session, new VM, etc.).

In [ ]:
import subprocess, sys, os

REPO_URL  = "https://github.com/tiensinh2/CMF_UNLearning_Posthoc.git"
REPO_NAME = "CMF_UNLearning_Posthoc"   # folder cloned into

# ── clone (skip if already present) ─────────────────────────────────────────
if not os.path.isdir(REPO_NAME):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    print("Clone complete.")
else:
    print(f"Repo already exists at ./{REPO_NAME} — skipping clone.")

# ── install dependencies ─────────────────────────────────────────────────────
_req = os.path.join(REPO_NAME, "requirements.txt")
if os.path.isfile(_req):
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", _req], check=True)
    print("Dependencies installed.")
else:
    # Minimal hard-coded fallback if requirements.txt is absent
    _pkgs = ["torch", "torchvision", "pytorch-lightning", "torchmetrics",
             "pyyaml", "pandas", "numpy", "timm"]
    print(f"requirements.txt not found — installing: {_pkgs}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _pkgs, check=True)

# ── add repo root to sys.path ────────────────────────────────────────────────
_NB_DIR = os.path.abspath(REPO_NAME)
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)

print(f"Repo root on sys.path: {_NB_DIR}")

---
## Cell 1 — Choose dataset  ✏️  ← the ONLY edit needed

Set `DATASET` to `"cifar10"` or `"cifar100"`.  
The notebook will automatically load the matching config file from `configs/`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  DATASET — the ONLY value you should change between experiments.        ║
# ║  "cifar10"  → loads configs/nb1_config_cifar10.yaml                    ║
# ║  "cifar100" → loads configs/nb1_config_cifar100.yaml                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

DATASET = "cifar10"   # ← change to "cifar100" for CIFAR-100

---
## Cell 2 — Imports

In [ ]:
import copy
import json
import random
import time
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional

import yaml

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR, SequentialLR

# ── project modules ──────────────────────────────────────────────────────────
from paper_hparams import GLOBAL_CFG, PRETRAIN_CFG, CKPT_DIRS
from train import train
from unlearn.cmf_weights import ModelModule
from utils import (
    get_dataset,
    get_retain_forget_partition,
    test,
    to_jsonable,
)

print("All imports OK.")

---
## Cell 3 — Load & resolve configuration

Selects `configs/nb1_config_{DATASET}.yaml` based on the `DATASET` variable
set in Cell 1, loads it, and merges with `paper_hparams.PRETRAIN_CFG` defaults.

In [ ]:
# ── select config file from DATASET variable ─────────────────────────────────
_CONFIG_MAP = {
    "cifar10":  "configs/nb1_config_cifar10.yaml",
    "cifar100": "configs/nb1_config_cifar100.yaml",
}
assert DATASET in _CONFIG_MAP, (
    f"Unknown DATASET '{DATASET}'. Choose from: {list(_CONFIG_MAP.keys())}"
)
_config_path = Path(_NB_DIR) / _CONFIG_MAP[DATASET]
assert _config_path.exists(), f"Config file not found: {_config_path}"
with open(_config_path, "r", encoding="utf-8") as _f:
    _cfg: Dict[str, Any] = yaml.safe_load(_f)
print(f"DATASET='{DATASET}' → loaded config: {_config_path}")

# ── merge: paper defaults first, then YAML overrides ────────────────────────
_hp = {**PRETRAIN_CFG}                      # start from paper defaults
_hp_overrides = _cfg.get("hparam_overrides") or {}
_hp.update(_hp_overrides)                   # intentional ablation overrides

# ── top-level protocol fields ────────────────────────────────────────────────
EXPERIMENT_NAME  = _cfg["experiment_name"]
SUFFIX           = _cfg.get("suffix", "")
DATASET          = _cfg["dataset"]
ARCH             = _cfg["arch"]
DATA_PATH        = _cfg.get("data_path",        GLOBAL_CFG["data_path"])

# ── Kaggle: redirect data_path to writable /kaggle/working/data ─────────────
# /kaggle/input/ is read-only; torchvision's download=True would crash there.
# If the Kaggle CIFAR dataset is attached, symlink its already-extracted
# folder into the writable path so no re-download is needed.
_KAGGLE_INPUT = Path("/kaggle/input")
if _KAGGLE_INPUT.exists():
    DATA_PATH = "/kaggle/working/data"
    _writable_data = Path(DATA_PATH)
    _writable_data.mkdir(parents=True, exist_ok=True)
    _CIFAR_FOLDER_MAP = {"cifar10": "cifar-10-batches-py", "cifar100": "cifar-100-python"}
    _expected_folder = _CIFAR_FOLDER_MAP.get(DATASET)
    if _expected_folder and not (_writable_data / _expected_folder).exists():
        import glob as _glob
        _matches = _glob.glob(f"/kaggle/input/**/{_expected_folder}", recursive=True)
        if _matches:
            (_writable_data / _expected_folder).symlink_to(_matches[0])
            print(f"[Kaggle] Symlinked {_matches[0]} -> {_writable_data / _expected_folder}")
        else:
            print(f"[Kaggle] WARNING: '{_expected_folder}' not found in /kaggle/input. "
                  "torchvision will attempt to download (may fail without internet).")
    print(f"[Kaggle] data_path overridden to: {DATA_PATH}")

SEEDS: List[int] = _cfg.get("SEEDS",             GLOBAL_CFG["SEEDS"])
TEST_MODE: bool  = bool(_cfg.get("TEST_MODE",    GLOBAL_CFG["TEST_MODE"]))
TEST_EPOCHS_SCALE: float = float(
    _cfg.get("TEST_EPOCHS_SCALE", GLOBAL_CFG["TEST_EPOCHS_SCALE"])
)

BATCH_SIZE      = int(_cfg.get("batch_size",      GLOBAL_CFG["batch_size"]))
TEST_BATCH_SIZE = int(_cfg.get("test_batch_size", GLOBAL_CFG["test_batch_size"]))
NUM_WORKERS     = int(_cfg.get("num_workers",     GLOBAL_CFG["num_workers"]))

# ── output directories ───────────────────────────────────────────────────────
PRETRAIN_DIR = Path(_cfg.get("pretrain_dir", CKPT_DIRS["pretrain"]))
SPLITS_DIR   = Path(_cfg.get("splits_dir",   CKPT_DIRS["splits"]))
RESULTS_DIR  = Path(_cfg.get("results_dir",  CKPT_DIRS["results"]))

PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── test-mode epoch scaling ──────────────────────────────────────────────────
if TEST_MODE:
    _hp["epochs"] = max(1, int(_hp["epochs"] * TEST_EPOCHS_SCALE))
    SUFFIX = (SUFFIX or "") + "_test"
    print(f"[TEST_MODE] epochs reduced to {_hp['epochs']}, suffix set to '{SUFFIX}'")

# ── forget_classes resolved after dataset load ───────────────────────────────
_forget_classes_cfg = _cfg.get("forget_classes")   # None → all classes

print("=" * 60)
print(f"Experiment : {EXPERIMENT_NAME}{SUFFIX}")
print(f"Dataset    : {DATASET}")
print(f"Arch       : {ARCH}")
print(f"Seeds      : {SEEDS}")
print(f"Test mode  : {TEST_MODE}")
print(f"Pretrain HP: {_hp}")
print(f"HP overrides applied: {_hp_overrides}")
print("=" * 60)

---
## Cell 3 — Build `args` namespace

Constructs the `SimpleNamespace` that the project's `utils.py` and
`unlearn/cmf_weights.py` expect.  All values come from the YAML config /
merged hyperparameter dict — never hardcoded here.

In [ ]:
import types

args = types.SimpleNamespace(
    # ── dataset / arch ─────────────────────────────────────────────
    dataset          = DATASET,
    arch             = ARCH,
    data_path        = DATA_PATH,
    train_transform  = True,       # always use augmentation for pre-training
    num_classes      = -1,         # filled in by get_dataset
    class_label_names= [],         # filled in by get_dataset

    # ── unlearn_method: must be "pre_train" to trigger the right code paths ──
    unlearn_method   = "pre_train",
    unlearn_class    = [],

    # ── CMF model flags (required by ModelModule) ───────────────────
    CMFClassifier    = True,
    remove_FC        = True,
    CMF_momentum     = float(_hp.get("CMF_momentum", 0.9)),
    temperature      = float(_hp.get("temperature",  1.0)),
    pretrained       = False,

    # ── optimiser / training ────────────────────────────────────────
    lr               = float(_hp["lr"]),
    momentum         = float(_hp["momentum"]),
    weight_decay     = float(_hp["weight_decay"]),
    nesterov         = bool(_hp["nesterov"]),
    epochs_or_steps  = int(_hp["epochs"]),

    # ── scheduler ───────────────────────────────────────────────────
    lr_scheduler     = _hp.get("scheduler", "cosine"),
    warmup_epochs    = int(_hp.get("warmup_epochs", 5)),
    min_lr           = 1e-5,
    patience         = int(_hp.get("early_stop_patience", 50)),

    # ── dataloader ──────────────────────────────────────────────────
    batch_size       = BATCH_SIZE,
    test_batch_size  = TEST_BATCH_SIZE,
    num_workers      = NUM_WORKERS,

    # ── misc ────────────────────────────────────────────────────────
    val_ratio        = 0.1,
    log_interval     = 100,
    dry_run          = False,
    save_model       = True,
    gpu_id           = 0,
    sub_set_mode     = False,
    sub_set_samples  = 10000,
)

print("args namespace ready.")

---
## Cell 4 — Device selection

In [ ]:
if torch.cuda.is_available():
    device = torch.device(f"cuda:{args.gpu_id}")
    torch.cuda.set_device(args.gpu_id)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

---
## Cell 5 — Load dataset (once)

The full dataset is loaded **once** and reused across all seed iterations.
Splitting into train/val is done per seed inside the training loop using a
seeded `torch.Generator` so splits are deterministic and reproducible.

In [ ]:
# get_dataset uses args.train_transform and args.dataset / args.data_path
train_dataset_full, test_dataset = get_dataset(args)

# Resolve forget_classes now that num_classes is known
if _forget_classes_cfg is None:
    FORGET_CLASSES: List[int] = list(range(args.num_classes))
else:
    FORGET_CLASSES = [int(c) for c in _forget_classes_cfg]

print(f"Train set size : {len(train_dataset_full)}")
print(f"Test  set size : {len(test_dataset)}")
print(f"num_classes    : {args.num_classes}")
print(f"forget_classes : {FORGET_CLASSES}")

---
## Cell 6 — Helper utilities

In [ ]:
def set_all_seeds(seed: int) -> None:
    """Pin every RNG source to `seed` for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ckpt_path_for_seed(seed: int) -> Path:
    """Canonical path for the pretrained checkpoint at a given seed."""
    return PRETRAIN_DIR / f"{EXPERIMENT_NAME}_seed{seed}{SUFFIX}.pt"


def log_path_for_seed(seed: int) -> Path:
    return PRETRAIN_DIR / f"{EXPERIMENT_NAME}_seed{seed}{SUFFIX}_trainlog.json"


def _split_train_val(dataset, val_ratio: float, seed: int):
    """Deterministic 90/10 train-val split using a seeded Generator."""
    total   = len(dataset)
    val_len = int(total * val_ratio)
    tr_len  = total - val_len
    g = torch.Generator().manual_seed(seed)
    all_idx = torch.randperm(total, generator=g).tolist()
    tr_idx, val_idx = all_idx[:tr_len], all_idx[tr_len:]

    if hasattr(dataset, "data"):  # CIFAR-style
        tr_ds  = copy.deepcopy(dataset)
        val_ds = copy.deepcopy(dataset)
        tr_ds.data  = dataset.data[tr_idx]
        val_ds.data = dataset.data[val_idx]
        tr_ds.targets  = [dataset.targets[i] for i in tr_idx]
        val_ds.targets = [dataset.targets[i] for i in val_idx]
    else:                          # ImageFolder-style
        tr_ds  = copy.deepcopy(dataset)
        val_ds = copy.deepcopy(dataset)
        tr_ds.samples  = [dataset.samples[i] for i in tr_idx]
        val_ds.samples = [dataset.samples[i] for i in val_idx]
        if hasattr(dataset, "imgs"):
            tr_ds.imgs  = tr_ds.samples
            val_ds.imgs = val_ds.samples
        if hasattr(dataset, "targets"):
            tr_ds.targets  = [dataset.targets[i] for i in tr_idx]
            val_ds.targets = [dataset.targets[i] for i in val_idx]
        else:
            tr_ds.targets  = [s[1] for s in tr_ds.samples]
            val_ds.targets = [s[1] for s in val_ds.samples]
    return tr_ds, val_ds


def _make_dataloaders(tr_ds, val_ds, test_ds):
    use_cuda = device.type == "cuda"
    tr_kw  = dict(batch_size=BATCH_SIZE,      num_workers=NUM_WORKERS,
                  pin_memory=use_cuda, shuffle=True)
    tst_kw = dict(batch_size=TEST_BATCH_SIZE, num_workers=NUM_WORKERS,
                  pin_memory=use_cuda, shuffle=False)
    return (
        torch.utils.data.DataLoader(tr_ds,   **tr_kw),
        torch.utils.data.DataLoader(val_ds,  **tst_kw),
        torch.utils.data.DataLoader(test_ds, **tst_kw),
    )


def _build_model() -> torch.nn.Module:
    """Construct ModelModule with CMFClassifier=True, remove_FC=True."""
    _args = copy.copy(args)
    _args.CMFClassifier = True
    _args.remove_FC     = True
    model = ModelModule(_args).to(device)
    # Mandatory guard: verify CMFweights attribute is present
    assert hasattr(model, "CMFweights"), (
        "ModelModule did not create CMFweights — check that "
        "CMFClassifier=True and remove_FC=True were accepted."
    )
    return model


print("Helpers defined.")

---
## Cell 7 — Full-model training loop (one checkpoint per seed)

For each seed:
- If a checkpoint already exists → skip training (load it).
- Otherwise → train from scratch with the paper schedule (cosine + warmup + early stopping),
  then save a self-describing checkpoint.

Any exception is caught, the full traceback is printed, and execution continues
to the next seed so split generation still runs for already-trained seeds.

In [ ]:
trained_seeds: Dict[int, Path] = {}   # seed → checkpoint path (populated below)
failed_seeds: List[int] = []

for seed in SEEDS:
    ckpt  = ckpt_path_for_seed(seed)
    logp  = log_path_for_seed(seed)
    print(f"\n{'='*60}")
    print(f"SEED {seed}  |  checkpoint: {ckpt}")
    print(f"{'='*60}")

    # ── checkpoint-skip ───────────────────────────────────────────────────────
    if ckpt.exists():
        print(f"[SKIP] Checkpoint exists — loading and skipping training.")
        model = _build_model()
        state = torch.load(str(ckpt), map_location=device)
        if "state_dict" in state:          # new canonical key
            model.load_state_dict(state["state_dict"])
        elif "model_state" in state:       # backward-compat key
            model.load_state_dict(state["model_state"])
        else:                              # legacy flat checkpoint
            model.load_state_dict(state)
        trained_seeds[seed] = ckpt
        continue

    # ── train from scratch ────────────────────────────────────────────────────
    try:
        t0 = time.time()
        set_all_seeds(seed)

        # split dataset deterministically (val used for early-stopping)
        tr_ds, val_ds = _split_train_val(train_dataset_full, args.val_ratio, seed)
        tr_loader, val_loader, test_loader = _make_dataloaders(tr_ds, val_ds, test_dataset)

        # fresh model
        model = _build_model()

        # optimiser
        optimizer = optim.SGD(
            model.parameters(),
            lr=args.lr,
            momentum=args.momentum,
            weight_decay=args.weight_decay,
            nesterov=args.nesterov,
        )

        # ── scheduler: warmup → cosine ────────────────────────────────────────
        warmup_ep    = max(0, args.warmup_epochs)
        total_ep     = args.epochs_or_steps
        cosine_ep    = max(1, total_ep - warmup_ep)

        warmup_sched = LambdaLR(
            optimizer,
            lr_lambda=lambda cur: min(1.0, (cur + 1) / max(1, warmup_ep))
            if warmup_ep > 0 else (lambda cur: 1.0)(cur),
        )
        cosine_sched = CosineAnnealingLR(optimizer, T_max=cosine_ep, eta_min=args.min_lr)
        scheduler    = SequentialLR(
            optimizer,
            schedulers=[warmup_sched, cosine_sched],
            milestones=[warmup_ep],
        )

        # ── training state ────────────────────────────────────────────────────
        best_val_acc     = 0.0
        best_epoch       = 0
        epochs_no_improv = 0
        # Early stopping must not fire during warmup or the 10 epochs
        # immediately after — the LR step-up at the warmup boundary causes a
        # transient accuracy dip that would otherwise consume patience budget.
        early_stop_start = warmup_ep + 10
        history          = {
            "epoch": [], "lr": [],
            "train_loss": [], "train_acc": [],
            "val_retain_acc": [], "val_forget_acc": [], "val_metric": [],
            "test_retain_acc": [], "test_forget_acc": [], "test_metric": [],
        }

        # temporary best-weight file (only kept until we build the final ckpt)
        _tmp_best = str(ckpt) + ".tmp_best"

        print(f"Training {total_ep} epochs, warmup={warmup_ep}, "
              f"patience={args.patience} (active from epoch {early_stop_start + 1})")

        for epoch in range(1, total_ep + 1):
            # Reset patience counter once after warmup ends so the LR jump
            # from warmup→cosine cannot consume patience budget.
            if epoch == warmup_ep + 1:
                epochs_no_improv = 0

            # args passed to train() must have log_interval and dry_run
            _tr_args = copy.copy(args)
            _tr_args.seed = seed
            tr_loss, tr_acc = train(_tr_args, model, device, tr_loader, optimizer, epoch, "descent")

            val_r, val_f, val_m = test(
                model, device, val_loader,
                [],                         # no forget class during pre-training eval
                args.class_label_names, args.num_classes,
                plot_cm=False, job_name="pre_train", verbose=False, set_name="Val"
            )
            tst_r, tst_f, tst_m = test(
                model, device, test_loader,
                [],
                args.class_label_names, args.num_classes,
                plot_cm=False, job_name="pre_train", verbose=False, set_name="Test"
            )

            cur_lr = float(optimizer.param_groups[0]["lr"])
            history["epoch"].append(epoch)
            history["lr"].append(cur_lr)
            history["train_loss"].append(float(tr_loss))
            history["train_acc"].append(float(tr_acc))
            history["val_retain_acc"].append(float(val_r))
            history["val_forget_acc"].append(float(val_f))
            history["val_metric"].append(val_m)
            history["test_retain_acc"].append(float(tst_r))
            history["test_forget_acc"].append(float(tst_f))
            history["test_metric"].append(tst_m)

            if val_r > best_val_acc:
                best_val_acc    = val_r
                best_epoch      = epoch
                epochs_no_improv = 0
                torch.save(model.state_dict(), _tmp_best)
                print(f"  epoch {epoch:4d} | val_acc {val_r:.4f} ← new best | lr {cur_lr:.6f}")
            else:
                epochs_no_improv += 1
                if epoch % 20 == 0:
                    print(f"  epoch {epoch:4d} | val_acc {val_r:.4f} | no-improv {epochs_no_improv}/{args.patience} | lr {cur_lr:.6f}")
                # Early stopping: only active after warmup + 10-epoch grace period
                if epoch > early_stop_start and epochs_no_improv >= args.patience:
                    print(f"  Early stop at epoch {epoch} (no improvement for {args.patience} epochs).")
                    break

            scheduler.step()

        # reload best weights
        if os.path.exists(_tmp_best):
            model.load_state_dict(torch.load(_tmp_best, map_location=device))
            os.remove(_tmp_best)

        runtime = time.time() - t0

        # final test evaluation
        final_r, final_f, final_m = test(
            model, device, test_loader,
            [],
            args.class_label_names, args.num_classes,
            plot_cm=False, job_name="pre_train", set_name="Final Test"
        )
        print(f"\nFinal test accuracy: {final_r:.4f}  (seed={seed}, best_epoch={best_epoch})")

        # ── self-describing checkpoint ────────────────────────────────────────
        _sd = model.state_dict()
        checkpoint = {
            "state_dict":     _sd,           # canonical key for downstream NB2-NB4c
            "model_state":    _sd,           # backward-compat alias
            "seed":           seed,
            "stage":          "pretrain",
            "config": {
                "experiment_name": EXPERIMENT_NAME,
                "suffix":          SUFFIX,
                "dataset":         DATASET,
                "arch":            ARCH,
                "CMFClassifier":   True,
                "remove_FC":       True,
                "hp":              {k: to_jsonable(v) for k, v in _hp.items()},
                "hp_overrides":    _hp_overrides,
            },
            "metrics": {
                "best_epoch":     best_epoch,
                "best_val_acc":   best_val_acc,
                "final_test_acc": float(final_r),
            },
            "wall_clock_minutes": round(runtime / 60, 3),
            "history":        history,
        }
        torch.save(checkpoint, str(ckpt))
        print(f"[SAVED] {ckpt}")

        # training log JSON (human-readable, no tensors)
        log_data = {k: v for k, v in checkpoint.items()
                    if k not in ("state_dict", "model_state")}
        with open(str(logp), "w") as _lf:
            json.dump(log_data, _lf, indent=2, default=to_jsonable)
        print(f"[LOG]   {logp}")

        trained_seeds[seed] = ckpt

    except Exception:  # catch-all so other seeds still run
        print(f"\n[ERROR] Training failed for seed {seed}:")
        traceback.print_exc()
        failed_seeds.append(seed)

print("\n" + "=" * 60)
print(f"Training complete. Successful seeds: {list(trained_seeds.keys())}")
if failed_seeds:
    print(f"FAILED seeds (no checkpoint produced): {failed_seeds}")

---
## Cell 8 — Generate whole-class-single splits

For every `(seed, forget_class)` pair this cell:
1. Partitions the **full** train set using `get_retain_forget_partition`.
2. Partitions the test set the same way.
3. Validates that the splits fully partition each dataset with no overlap.
4. Saves a JSON file `splits_dir/<experiment>_seed<N>_fc<C>.json` containing
   all four index lists plus audit metadata.

Splits are generated even for seeds that failed training so that they can be
used once a corrected checkpoint is produced.

In [ ]:
split_records: List[Dict[str, Any]] = []  # accumulates rows for summary CSV
failed_splits: List[tuple]          = []

for seed in SEEDS:
    set_all_seeds(seed)  # ensure index generation is deterministic

    for fc in FORGET_CLASSES:
        split_file = SPLITS_DIR / f"{EXPERIMENT_NAME}_seed{seed}_fc{fc}{SUFFIX}.json"

        if split_file.exists():
            print(f"[SKIP] Split exists: {split_file.name}")
            # still read metadata for the summary CSV
            try:
                with open(str(split_file)) as _sf:
                    _meta = json.load(_sf)
                split_records.append({
                    "seed":            seed,
                    "forget_class":    fc,
                    "n_forget_train":  _meta["n_forget_train"],
                    "n_retain_train":  _meta["n_retain_train"],
                    "n_forget_test":   _meta["n_forget_test"],
                    "n_retain_test":   _meta["n_retain_test"],
                    "valid":           _meta.get("valid", True),
                    "split_file":      str(split_file),
                })
            except Exception:
                pass
            continue

        try:
            # ── whole-class-single: one class at a time ───────────────────────
            _split_args = copy.copy(args)
            _split_args.unlearn_class = [fc]

            _, _, retain_tr_idx, forget_tr_idx = get_retain_forget_partition(
                _split_args, train_dataset_full, [fc], return_ind=True
            )
            _, _, retain_tst_idx, forget_tst_idx = get_retain_forget_partition(
                _split_args, test_dataset, [fc], return_ind=True
            )

            # ── validation: full partition, no overlap ────────────────────────
            n_train = len(train_dataset_full)
            n_test  = len(test_dataset)

            _tr_union = set(retain_tr_idx) | set(forget_tr_idx)
            _tr_inter = set(retain_tr_idx) & set(forget_tr_idx)
            _tst_union = set(retain_tst_idx) | set(forget_tst_idx)
            _tst_inter = set(retain_tst_idx) & set(forget_tst_idx)

            valid = True
            if len(_tr_union) != n_train:
                print(f"[WARN] seed={seed} fc={fc}: train split does not cover all {n_train} samples (got {len(_tr_union)})")
                valid = False
            if _tr_inter:
                print(f"[WARN] seed={seed} fc={fc}: train split has {len(_tr_inter)} overlapping indices")
                valid = False
            if len(_tst_union) != n_test:
                print(f"[WARN] seed={seed} fc={fc}: test split does not cover all {n_test} samples (got {len(_tst_union)})")
                valid = False
            if _tst_inter:
                print(f"[WARN] seed={seed} fc={fc}: test split has {len(_tst_inter)} overlapping indices")
                valid = False

            assert valid, (
                f"Split validation failed for seed={seed}, forget_class={fc}. "
                "Check warnings above."
            )

            # ── save ─────────────────────────────────────────────────────────
            split_payload = {
                "seed":              seed,
                "forget_class":      fc,
                "protocol":          "whole_class_single",
                "dataset":           DATASET,
                "experiment_name":   EXPERIMENT_NAME,
                "suffix":            SUFFIX,
                "n_forget_train":    len(forget_tr_idx),
                "n_retain_train":    len(retain_tr_idx),
                "n_forget_test":     len(forget_tst_idx),
                "n_retain_test":     len(retain_tst_idx),
                "valid":             valid,
                "forget_train_idx":  forget_tr_idx,
                "retain_train_idx":  retain_tr_idx,
                "forget_test_idx":   forget_tst_idx,
                "retain_test_idx":   retain_tst_idx,
            }

            with open(str(split_file), "w") as _sf:
                json.dump(split_payload, _sf)

            split_records.append({
                "seed":           seed,
                "forget_class":   fc,
                "n_forget_train": len(forget_tr_idx),
                "n_retain_train": len(retain_tr_idx),
                "n_forget_test":  len(forget_tst_idx),
                "n_retain_test":  len(retain_tst_idx),
                "valid":          valid,
                "split_file":     str(split_file),
            })
            print(f"[SAVED] seed={seed} fc={fc} "
                  f"forget_tr={len(forget_tr_idx)} retain_tr={len(retain_tr_idx)} "
                  f"forget_tst={len(forget_tst_idx)} retain_tst={len(retain_tst_idx)}")

        except Exception:
            print(f"\n[ERROR] Split generation failed for seed={seed} fc={fc}:")
            traceback.print_exc()
            failed_splits.append((seed, fc))

print("\nSplit generation complete.")
if failed_splits:
    print(f"FAILED (seed, fc) pairs: {failed_splits}")

---
## Cell 9 — Save split summary CSV

Writes `results_dir/<experiment_name><suffix>_split_summary.csv`.
Downstream notebooks can load this file to validate they are using the
correct splits.

In [ ]:
if split_records:
    summary_df = pd.DataFrame(split_records)
    summary_path = RESULTS_DIR / f"{EXPERIMENT_NAME}{SUFFIX}_split_summary.csv"
    summary_df.to_csv(str(summary_path), index=False)
    print(f"Split summary saved → {summary_path}")
    display(summary_df)
else:
    print("No split records to summarise (all were skipped or failed).")

---
## Cell 10 — Final output manifest

Print a concise summary of all artefacts produced so the notebook is
self-documenting when run as a report.

In [ ]:
print("="*60)
print("NB1 — OUTPUT MANIFEST")
print("="*60)

print("\n[Pretrained checkpoints]")
for seed, path in trained_seeds.items():
    size_mb = path.stat().st_size / 1_048_576 if path.exists() else 0
    print(f"  seed={seed}  →  {path}  ({size_mb:.1f} MB)")

print("\n[Split files]")
split_files_all = sorted(SPLITS_DIR.glob(f"{EXPERIMENT_NAME}*{SUFFIX}.json"))
for p in split_files_all:
    print(f"  {p}")

print("\n[Summary CSV]")
_csv = RESULTS_DIR / f"{EXPERIMENT_NAME}{SUFFIX}_split_summary.csv"
print(f"  {_csv}" if _csv.exists() else "  (not produced)")

if failed_seeds:
    print(f"\n[WARNING] Training failed for seeds: {failed_seeds}")
if failed_splits:
    print(f"[WARNING] Split generation failed for: {failed_splits}")

print("\nNB1 complete. Downstream notebooks should consume the artefacts above.")